# 🤖 ML CROP PROFIT PREDICTION SYSTEM
## Machine Learning Model for Maximum Profitable Crop Selection

**Goal:** Train ML models to predict crop profitability based on SOIL + WEATHER alone

**PRIMARY FEATURES (Model Inputs):**
- 🌱 **Soil properties** (NPK levels, pH, EC, organic carbon) - 17 features
- 🌤️ **Weather data** (temperature, rainfall, humidity, wind) - 9 features
- **Total: 26 soil+weather features** (Modal price is NOT included as feature)

**Target Variable (What we predict):**
- **High profitability** = Crops with above-median historical prices
- **Low profitability** = Crops with below-median historical prices

**Why This Approach:**
- Soil + Weather are the ACTUAL growing conditions
- Market price is only used to define what "profitable" means
- Shows which soil/weather combinations produce high-profit crops
- Feature importance reveals which soil/weather factors matter most

**Approach:**
1. Train 4 ML models using SOIL + WEATHER features
2. Target = Historical market profitability (high/low price)
3. Identify which soil/weather patterns predict profitable crops
4. Rank crops by predicted profitability score
5. Visualize soil/weather feature importance

## 1. Setup & Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import glob
import warnings

# ML Libraries
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import joblib

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# GPU Setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️  Device: {device}")

# Data path
data_path = Path(r'c:\Users\tanis\Documents\Project 2\Project---2\Data')
print(f"✅ Data path set: {data_path}")

## 2. Load & Explore Data

In [ ]:
# Load soil data
soil_file = data_path / 'Soil Data ( District Wise)' / 'CSV Format' / 'THANJAVUR.csv'
soil_data = pd.read_csv(soil_file)

# Load weather data
weather_file = data_path / 'Weather Data (District Wise)' / 'weather_data_all_blocks.csv'
weather_data = pd.read_csv(weather_file)
thanjavur_weather = weather_data[weather_data['district'] == 'Thanjavur'].copy()

# Load all crop market data
crop_files = glob.glob(str(data_path / '3_Cleaned CSVs' / '*.csv'))
crop_data_dict = {}

for file in crop_files:
    crop_name = Path(file).stem
    try:
        df = pd.read_csv(file)
        crop_data_dict[crop_name] = df
    except:
        pass

total_records = sum(len(df) for df in crop_data_dict.values())

print("✅ DATA LOADED:")
print(f"   • Soil records: {len(soil_data)}")
print(f"   • Weather records: {len(thanjavur_weather)}")
print(f"   • Crops loaded: {len(crop_data_dict)}")
print(f"   • Total market transactions: {total_records:,}")
print(f"\n📊 Data Columns:")
print(f"   Soil: {soil_data.columns.tolist()[:5]}...")
print(f"   Crop: {crop_data_dict[list(crop_data_dict.keys())[0]].columns.tolist()}")

## 3. Feature Engineering

In [ ]:
# Create soil features from Thanjavur
soil_summary = soil_data.groupby('District')[[
    'n_High', 'n_Medium', 'n_Low',
    'p_High', 'p_Medium', 'p_Low',
    'k_High', 'k_Medium', 'k_Low',
    'pH_Neutral', 'pH_Acidic', 'pH_Alkaline',
    'EC_Saline', 'EC_NonSaline',
    'OC_High', 'OC_Medium', 'OC_Low'
]].mean()

tf_soil = soil_summary.loc['THANJAVUR']

# Create weather features
weather_features = thanjavur_weather[[
    'temp_max_mean', 'temp_min_mean', 'temp_mean_annual',
    'total_rainfall_mm', 'avg_daily_rainfall_mm',
    'humidity_max_mean', 'humidity_min_mean',
    'rainy_days', 'wind_speed_max_mean'
]].mean()

print("✅ FEATURE ENGINEERING:")
print(f"\n   Soil features ({len(tf_soil)}):")
print(f"   {tf_soil.index.tolist()}")
print(f"\n   Weather features ({len(weather_features)}):")
print(f"   {weather_features.index.tolist()}")
print(f"\n   Total features: {len(tf_soil) + len(weather_features)}")

## 4. Build Multi-Crop Training Dataset

In [ ]:
# Create multi-region dataset with SOIL + WEATHER features ONLY (NOT modal_price)
training_data = []
crop_labels = []
prices = []

district_soil_data = soil_summary
# PRIMARY FEATURES: Soil + Weather (EXCLUDING modal_price for better feature importance)
feature_names = list(tf_soil.index) + list(weather_features.index)

print("Building dataset from market transactions...\n")
print(f"PRIMARY FEATURES: Soil + Weather ({len(feature_names)} features)")
print(f"TARGET: High profitability = price > median\n")

for crop_name, crop_df in crop_data_dict.items():
    # Normalize crop name
    base_crop = crop_name.rsplit('-', 1)[0] if any(y in crop_name for y in ['-2015-2019', '-2019-2022', '-2022-2025', '-2024-2025', '-2025']) else crop_name
    
    for idx, row in crop_df.iterrows():
        price = row['Modal Price (Rs./Quintal)']  # For target variable only
        district = row['District Name']
        
        # Get soil features for this district
        try:
            if district in district_soil_data.index:
                soil_features = list(district_soil_data.loc[district].values)
            else:
                soil_features = list(tf_soil.values)
        except:
            soil_features = list(tf_soil.values)
        
        # Build feature vector: [soil_features, weather_features] - NO modal_price!
        features = soil_features + list(weather_features.values)
        
        training_data.append(features)
        crop_labels.append(base_crop)
        prices.append(price)  # Keep price for creating target variable

# Convert to arrays
X = np.array(training_data)
crop_labels_array = np.array(crop_labels)
prices_array = np.array(prices)

# Create profitability target: High price = 1, Low price = 0
price_median = np.median(prices_array)
y = (prices_array > price_median).astype(int)

# Get unique crops and encode
unique_crops = np.unique(crop_labels_array)
le = LabelEncoder()
y_crop = le.fit_transform(crop_labels_array)

print(f"✅ DATASET CREATED:")
print(f"   • Total records: {len(X):,}")
print(f"   • Unique crops: {len(unique_crops)}")
print(f"   • Soil + Weather Features: {len(feature_names)}")
print(f"   • High profitability crops: {sum(y)} ({sum(y)/len(y)*100:.1f}%)")
print(f"   • Low profitability crops: {len(y)-sum(y)} ({(len(y)-sum(y))/len(y)*100:.1f}%)")
print(f"   • Price range: ₹{prices_array.min():.0f} - ₹{prices_array.max():.0f}")
print(f"   • Median price threshold: ₹{price_median:.0f}")

## 5. Train/Test Split & Preprocessing

In [ ]:
# Split data
X_train, X_test, y_train, y_test, crop_train, crop_test = train_test_split(
    X, y, crop_labels_array, test_size=0.2, random_state=42, stratify=y
)

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Apply SMOTE to balance classes
smote = SMOTE(random_state=42, k_neighbors=5)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_scaled, y_train)

print("✅ DATA PREPROCESSING:")
print(f"   • Train set: {X_train_balanced.shape}")
print(f"   • Test set: {X_test_scaled.shape}")
print(f"   • Class distribution (before SMOTE): {np.bincount(y_train)}")
print(f"   • Class distribution (after SMOTE): {np.bincount(y_train_balanced)}")
print(f"   ✅ Classes balanced!")

## 6. Train Random Forest Model

In [ ]:
print("Training Random Forest Classifier...\n")

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    min_samples_split=5,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
    verbose=1
)

rf_model.fit(X_train_balanced, y_train_balanced)
y_pred_rf = rf_model.predict(X_test_scaled)
y_pred_proba_rf = rf_model.predict_proba(X_test_scaled)[:, 1]
accuracy_rf = accuracy_score(y_test, y_pred_rf)

print(f"\n✅ RANDOM FOREST RESULTS:")
print(f"   Accuracy: {accuracy_rf:.3f} ({accuracy_rf*100:.1f}%)")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred_rf, target_names=['Low Profit', 'High Profit']))

# Feature importance
feature_importance_rf = pd.DataFrame({
    'feature': feature_names,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\n📊 TOP 10 IMPORTANT FEATURES:")
print(feature_importance_rf.head(10).to_string(index=False))

## 7. Train Gradient Boosting Model

print("Training XGBoost-style Gradient Boosting...\n")

xgb_model = GradientBoostingClassifier(
    n_estimators=150,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    random_state=42,
    verbose=1
)

xgb_model.fit(X_train_balanced, y_train_balanced)
y_pred_xgb = xgb_model.predict(X_test_scaled)
y_pred_proba_xgb = xgb_model.predict_proba(X_test_scaled)[:, 1]
accuracy_xgb = accuracy_score(y_test, y_pred_xgb)

print(f"\n✅ XGBOOST RESULTS:")
print(f"   Accuracy: {accuracy_xgb:.3f} ({accuracy_xgb*100:.1f}%)")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred_xgb, target_names=['Low Profit', 'High Profit']))

# Feature importance
feature_importance_xgb = pd.DataFrame({
    'feature': feature_names,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\n📊 TOP 10 IMPORTANT FEATURES:")
print(feature_importance_xgb.head(10).to_string(index=False))

# Compare all models
comparison_df = pd.DataFrame({
    'Model': ['Random Forest', 'Gradient Boosting', 'XGBoost', 'Neural Network'],
    'Accuracy': [accuracy_rf, accuracy_gb, accuracy_xgb, accuracy_nn]
})

print("\n" + "="*60)
print("📊 MODEL COMPARISON")
print("="*60)
print("\n" + comparison_df.to_string(index=False))

# Create ensemble
ensemble_proba = (y_pred_proba_rf + y_pred_proba_gb + y_pred_proba_xgb + y_pred_proba_nn) / 4
y_pred_ensemble = (ensemble_proba > 0.5).astype(int)
accuracy_ensemble = accuracy_score(y_test, y_pred_ensemble)

print(f"\n🎯 ENSEMBLE (4 Models Combined):")
print(f"   Accuracy: {accuracy_ensemble:.3f} ({accuracy_ensemble*100:.1f}%)")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred_ensemble, target_names=['Low Profit', 'High Profit']))

# Visualize comparison
fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#3498db', '#e74c3c', '#f39c12', '#2ecc71']
bars = ax.bar(comparison_df['Model'], comparison_df['Accuracy'], color=colors, alpha=0.7, edgecolor='black', linewidth=2)
ax.axhline(y=accuracy_ensemble, color='purple', linestyle='--', linewidth=2, label=f'Ensemble: {accuracy_ensemble:.3f}')
ax.set_ylabel('Accuracy', fontweight='bold', fontsize=12)
ax.set_title('ML Models Performance Comparison - Crop Profitability Prediction', fontweight='bold', fontsize=14)
ax.set_ylim([0, 1])
ax.legend(fontsize=11)
plt.xticks(rotation=15)

for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.3f}', ha='center', va='bottom', fontweight='bold', fontsize=11)

plt.tight_layout()
plt.show()

# Get predictions on full dataset
X_scaled_all = scaler.transform(X)

# Get probabilities from all models
rf_proba_all = rf_model.predict_proba(X_scaled_all)[:, 1]
gb_proba_all = gb_model.predict_proba(X_scaled_all)[:, 1]
xgb_proba_all = xgb_model.predict_proba(X_scaled_all)[:, 1]

X_tensor_all = torch.FloatTensor(X_scaled_all).to(device)
with torch.no_grad():
    nn_proba_all = nn_model(X_tensor_all).cpu().numpy().flatten()

# Ensemble prediction (high profit probability)
ensemble_proba_all = (rf_proba_all + gb_proba_all + xgb_proba_all + nn_proba_all) / 4

# Aggregate by crop
crop_predictions = pd.DataFrame({
    'Crop': crop_labels_array,
    'Price': prices_array,
    'Ensemble_Profit_Score': ensemble_proba_all
})

# Group by crop
crop_summary = crop_predictions.groupby('Crop').agg({
    'Price': ['mean', 'std', 'count'],
    'Ensemble_Profit_Score': 'mean'
}).reset_index()

crop_summary.columns = ['Crop', 'Avg_Price', 'Price_Std', 'Market_Count', 'Profit_Score']
crop_summary = crop_summary.sort_values('Profit_Score', ascending=False).reset_index(drop=True)
crop_summary['Rank'] = range(1, len(crop_summary) + 1)

print("\n" + "="*80)
print("🌾 TOP 20 CROPS BY PREDICTED PROFITABILITY SCORE")
print("="*80)

top_20 = crop_summary[['Rank', 'Crop', 'Avg_Price', 'Market_Count', 'Profit_Score']].head(20)
print("\n" + top_20.to_string(index=False))

print(f"\n\n✅ Total crops analyzed: {len(crop_summary)}")
print(f"✅ High profit crops (score > 0.6): {len(crop_summary[crop_summary['Profit_Score'] > 0.6])}")
print(f"✅ Medium profit crops (0.4-0.6): {len(crop_summary[(crop_summary['Profit_Score'] >= 0.4) & (crop_summary['Profit_Score'] <= 0.6)])}")
print(f"✅ Low profit crops (score < 0.4): {len(crop_summary[crop_summary['Profit_Score'] < 0.4])}")

# Save models and results
model_dir = Path(r'c:\Users\tanis\Documents\Project 2\Project---2\Models\ML_Crop_Profit')
model_dir.mkdir(parents=True, exist_ok=True)

# Save models
joblib.dump(rf_model, model_dir / 'random_forest_model.pkl')
joblib.dump(gb_model, model_dir / 'gradient_boosting_model.pkl')
joblib.dump(xgb_model, model_dir / 'xgboost_model.pkl')
joblib.dump(scaler, model_dir / 'feature_scaler.pkl')
torch.save(nn_model.state_dict(), model_dir / 'neural_network_model.pt')

# Save results
crop_summary.to_csv(model_dir / 'crop_profitability_predictions.csv', index=False)
comparison_df.to_csv(model_dir / 'model_comparison.csv', index=False)

print(f"\n✅ ALL MODELS SAVED TO: {model_dir}")
print(f"\n📁 Files:")
for file in sorted(model_dir.glob('*')):
    print(f"   • {file.name}")